# Day 031 — Exercise 3: resilient_step

**What you'll build:** `resilient_step(name, fn, max_attempts=3, base_delay=1.0, backoff=2.0) -> dict` — combines retry logic with the Day 30 step result schema. Tries `fn()` up to `max_attempts` times; returns an ok dict on first success or an error dict after all failures. Adds an `attempts` key (int) to the schema.

**Why it matters:** resilient_step is a drop-in upgrade over run_step (Day 30). Any step in a pipeline can be hardened by switching from run_step to resilient_step without changing the caller — same schema, extra resilience.

In [ ]:
import time

## Provided: RETRY_IMPL is used internally — implement the loop directly

## Your Implementation

In [ ]:
def resilient_step(
    name: str,
    fn,
    max_attempts: int = 3,
    base_delay: float = 1.0,
    backoff: float = 2.0,
) -> dict:
    """
    Run fn() with retry; return a step result dict.

    Returns dict with keys: name, status ('ok'|'error'), result, error,
    duration_s, attempts (int — how many tries were needed).
    Never raises.
    """
    # TODO: start = time.time(); last_error = None
    # TODO: for attempt in range(max_attempts):
    #     try: result = fn(); return ok-dict (attempts=attempt+1)
    #     except Exception as e:
    #         last_error = e
    #         if attempt < max_attempts - 1: time.sleep(base_delay*(backoff**attempt))
    # TODO: return error-dict (attempts=max_attempts)
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Check 1: defined
    try:
        assert 'resilient_step' in globals()
        passed += 1; print('\u2705 Check 1: resilient_step defined')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}')
        return

    # Check 2: immediate success → status=ok, attempts=1, all 6 keys
    try:
        r = resilient_step('compute', lambda: 7 * 6, max_attempts=3, base_delay=0.0)
        for k in ('name', 'status', 'result', 'error', 'duration_s', 'attempts'):
            assert k in r, f'missing key: {k}'
        assert r['status']   == 'ok', f"status: {r['status']!r}"
        assert r['result']   == 42,   f"result: {r['result']!r}"
        assert r['attempts'] == 1,    f"attempts: {r['attempts']}"
        passed += 1; print('\u2705 Check 2: immediate success → status=ok, attempts=1')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: fn fails once then succeeds → attempts=2
    try:
        _n = [0]
        def _flaky2():
            _n[0] += 1
            if _n[0] < 2:
                raise RuntimeError('transient')
            return 'ok'
        r = resilient_step('fetch', _flaky2, max_attempts=3, base_delay=0.0)
        assert r['status']   == 'ok',  f"status: {r['status']!r}"
        assert r['attempts'] == 2,     f"attempts should be 2, got {r['attempts']}"
        passed += 1; print('\u2705 Check 3: 1 failure then success → attempts=2')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: always fails → status=error, attempts=max_attempts
    try:
        def _fail(): raise ValueError('permanent')
        r = resilient_step('bad', _fail, max_attempts=3, base_delay=0.0)
        assert r['status']   == 'error',    f"status: {r['status']!r}"
        assert r['result']   is None,       f"result should be None: {r['result']!r}"
        assert r['attempts'] == 3,          f"attempts: {r['attempts']}"
        assert 'permanent'   in r['error'], f"error: {r['error']!r}"
        passed += 1; print('\u2705 Check 4: all-fail → status=error, attempts=max_attempts')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: resilient_step never raises
    try:
        r = resilient_step('bomb', lambda: 1/0, max_attempts=2, base_delay=0.0)
        assert isinstance(r, dict), 'should return dict, not raise'
        assert r['status'] == 'error'
        passed += 1; print('\u2705 Check 5: never raises — error captured in dict')
    except Exception as e:
        print(f'\u274c Check 5: resilient_step raised unexpectedly: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def resilient_step(
    name: str,
    fn,
    max_attempts: int = 3,
    base_delay: float = 1.0,
    backoff: float = 2.0,
) -> dict:
    start      = time.time()
    last_error = None
    for attempt in range(max_attempts):
        try:
            result = fn()
            return {
                "name":       name,
                "status":     "ok",
                "result":     result,
                "error":      None,
                "duration_s": round(time.time() - start, 3),
                "attempts":   attempt + 1,
            }
        except Exception as e:
            last_error = e
            if attempt < max_attempts - 1:
                time.sleep(base_delay * (backoff ** attempt))
    return {
        "name":       name,
        "status":     "error",
        "result":     None,
        "error":      str(last_error),
        "duration_s": round(time.time() - start, 3),
        "attempts":   max_attempts,
    }
```

</details>